# Homework 21: Attention as Learned Retrieval

**Audience.** Students who understand matrix multiplication, softmax, and the token embeddings from Homework 20.

**Prerequisites.** PyTorch tensors; rows and columns of a matrix; dot products; softmax; `nn.Sequential` models.

**Learning goals.** By the end, you will be able to:

- interpret queries, keys, and values;
- compute scaled dot-product attention;
- explain why attention weights sum to one;
- relate a custom `nn.Module` to a familiar `nn.Sequential` model;
- trace the tensor shapes in a single self-attention head.


## Outline

1. One query retrieves from several key/value pairs
2. Matrix attention processes many queries at once
3. Move from `nn.Sequential` to a custom `nn.Module`
4. Learned projections create queries, keys, and values
5. Notebook checkpoints


In [1]:
# S1: Imports and reproducibility
import math
import torch
from torch import nn

_ = torch.manual_seed(158)
torch.set_printoptions(precision=4, sci_mode=False)


## 1. One query, three memories

Think of a query as a request, each key as a description of an available memory, and each value as the information stored in that memory. A large query–key dot product gives the corresponding value more weight.


In [2]:
# S2: Scaled dot-product attention for one query
query = torch.tensor([1.0, 1.0])
keys = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])
values = torch.tensor([
    [10.0, 0.0],
    [0.0, 10.0],
    [6.0, 6.0],
])

scores = query @ keys.T / math.sqrt(keys.shape[1])
weights = torch.softmax(scores, dim=0)
retrieved = weights @ values

print("scores:", scores)
print("weights:", weights)
print("weight sum:", weights.sum())
print("retrieved value:", retrieved)


scores: tensor([0.7071, 0.7071, 1.4142])
weights: tensor([0.2483, 0.2483, 0.5035])
weight sum: tensor(1.)
retrieved value: tensor([5.5035, 5.5035])


Softmax makes all weights nonnegative and normalizes them to sum to one, so the result is a weighted average of the value rows. The scale factor `sqrt(key_dimension)` prevents dot products from growing too large merely because the vectors have many coordinates.


In [3]:
# S3: Changing a value does not change the attention weights
changed_values = values.clone()
changed_values[2] = torch.tensor([30.0, -10.0])
retrieved_after_value_change = weights @ changed_values

print("weights are unchanged:", weights)
print("old result:", retrieved)
print("new result:", retrieved_after_value_change)


weights are unchanged: tensor([0.2483, 0.2483, 0.5035])
old result: tensor([5.5035, 5.5035])
new result: tensor([17.5872, -2.5523])


The keys decide *where to look*. The values decide *what is retrieved*. Changing a value affects the output but not the weights; changing a key can affect both the weights and the output.


## 2. Self-attention

In self-attention, every sequence position supplies a query, key, and value. Matrix multiplication computes every query–key score at once.


In [4]:
# S4: Four token vectors and fixed projection matrices
token_vectors = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 1.0, 0.0],
    [1.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.0],
])

W_query = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.5, 0.5],
    [0.0, 0.0],
])
W_key = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.5, 0.5],
    [0.0, 0.0],
])
W_value = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [-1.0, 1.0],
])

Q = token_vectors @ W_query
K = token_vectors @ W_key
V = token_vectors @ W_value

print("Q shape:", tuple(Q.shape))
print("K shape:", tuple(K.shape))
print("V shape:", tuple(V.shape))


Q shape: (4, 2)
K shape: (4, 2)
V shape: (4, 2)


In [5]:
# S5: Every row is now a query
all_scores = Q @ K.T / math.sqrt(K.shape[1])
attention_weights = torch.softmax(all_scores, dim=-1)
self_attention_output = attention_weights @ V

print("score shape:", tuple(all_scores.shape))
print("attention weights:\n", attention_weights)
print("row sums:", attention_weights.sum(dim=-1))
print("output shape:", tuple(self_attention_output.shape))


score shape: (4, 4)
attention weights:
 tensor([[0.3935, 0.1940, 0.2763, 0.1362],
        [0.1940, 0.3935, 0.2763, 0.1362],
        [0.2863, 0.2863, 0.2863, 0.1412],
        [0.2701, 0.2701, 0.2701, 0.1897]])
row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000])
output shape: (4, 2)


Row `i` contains the weights used by position `i`. Column `j` tells how much position `i` retrieves from position `j`. At this stage, every position can look both backward and forward. We will correct that for text generation in the next homework.


## 3. From `nn.Sequential` to a custom module

`nn.Sequential` is already a subclass of `nn.Module`. It is a convenient module whose `forward` computation is fixed: send one tensor through each listed layer in order. A custom module uses the same PyTorch machinery, but we write the data flow ourselves.

Start with a familiar two-layer network. Before running S6, predict its output shape and whether `isinstance(sequential_mlp, nn.Module)` will be true or false.


In [6]:
# S6: A familiar network written with Sequential
sequential_mlp = nn.Sequential(
    nn.Linear(4, 6),
    nn.ReLU(),
    nn.Linear(6, 4),
)
bridge_input = torch.tensor([[1.0, -1.0, 0.5, 2.0]])
sequential_output = sequential_mlp(bridge_input)

print("Sequential is an nn.Module:", isinstance(sequential_mlp, nn.Module))
print("input shape:", tuple(bridge_input.shape))
print("output shape:", tuple(sequential_output.shape))


Sequential is an nn.Module: True
input shape: (1, 4)
output shape: (1, 4)


The custom version has four pieces worth recognizing:

1. `class TinyMLP(nn.Module)` says that this object is a PyTorch module.
2. `super().__init__()` initializes PyTorch's module bookkeeping.
3. Layers assigned to `self` are registered, so their parameters appear in `model.parameters()` and reach the optimizer.
4. `forward` defines the computation. Calling `custom_mlp(x)` lets PyTorch invoke `forward` while preserving hooks and other module behavior; normally, do not call `forward` yourself.

For a fair comparison, S7 copies the two linear layers' weights from S6. The two implementations should therefore compute exactly the same function.


In [7]:
# S7: The same network written as a custom Module
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.first = nn.Linear(4, 6)
        self.activation = nn.ReLU()
        self.second = nn.Linear(6, 4)

    def forward(self, x):
        hidden = self.activation(self.first(x))
        return self.second(hidden)

custom_mlp = TinyMLP()
custom_mlp.first.load_state_dict(sequential_mlp[0].state_dict())
custom_mlp.second.load_state_dict(sequential_mlp[2].state_dict())
custom_output = custom_mlp(bridge_input)

bridge_outputs_match = torch.allclose(sequential_output, custom_output)
custom_parameter_names = [name for name, _ in custom_mlp.named_parameters()]
custom_parameter_count = sum(parameter.numel() for parameter in custom_mlp.parameters())

print("outputs match:", bridge_outputs_match)
print("registered parameter names:", custom_parameter_names)
print("number of scalar parameters:", custom_parameter_count)


outputs match: True
registered parameter names: ['first.weight', 'first.bias', 'second.weight', 'second.bias']
number of scalar parameters: 58


The ReLU in S7 is a registered child module, but it has no trainable tensors, so it contributes no names to `named_parameters()`. Subclassing does not replace `Sequential`: custom modules often contain Sequential pieces. It becomes necessary when the computation branches, merges, loops, or returns extra information. S8 contains a Sequential transformation, adds a residual path around it, and returns both the final output and the change. An ordinary Sequential chain does not express that routing by itself.


In [8]:
# S8: Custom data flow around a Sequential subnetwork
class ResidualMLP(nn.Module):
    def __init__(self, dimension, hidden_dimension):
        super().__init__()
        self.transform = nn.Sequential(
            nn.Linear(dimension, hidden_dimension),
            nn.ReLU(),
            nn.Linear(hidden_dimension, dimension),
        )

    def forward(self, x):
        change = self.transform(x)
        output = x + change
        return output, change

residual_mlp = ResidualMLP(dimension=4, hidden_dimension=6)
residual_output, residual_change = residual_mlp(bridge_input)
residual_relation_holds = torch.allclose(
    residual_output, bridge_input + residual_change
)

print("output shape:", tuple(residual_output.shape))
print("output equals input + change:", residual_relation_holds)


output shape: (1, 4)
output equals input + change: True


## 4. A learned attention head

Attention needs custom data flow for three parallel projections, matrix multiplication, and two returned tensors. The fixed matrices from S4 become registered, trainable `nn.Linear` layers below. The class performs exactly the same attention operations as S4–S5, with a batch dimension added.


In [9]:
# S9: One learned, unmasked self-attention head
class SelfAttentionHead(nn.Module):
    def __init__(self, input_dimension, head_dimension):
        super().__init__()
        self.head_dimension = head_dimension
        self.query = nn.Linear(input_dimension, head_dimension, bias=False)
        self.key = nn.Linear(input_dimension, head_dimension, bias=False)
        self.value = nn.Linear(input_dimension, head_dimension, bias=False)

    def forward(self, x):
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dimension)
        weights = torch.softmax(scores, dim=-1)
        output = weights @ v
        return output, weights

head = SelfAttentionHead(input_dimension=4, head_dimension=2)
batched_tokens = token_vectors.unsqueeze(0)
learned_output, learned_weights = head(batched_tokens)

print("input shape:", tuple(batched_tokens.shape))
print("weight shape:", tuple(learned_weights.shape))
print("output shape:", tuple(learned_output.shape))


input shape: (1, 4, 4)
weight shape: (1, 4, 4)
output shape: (1, 4, 2)


## Notebook checkpoints

Trace S2–S9 before running this cell. For the bridge, predict which parameters PyTorch registers and why the two MLP outputs match. For attention, pay particular attention to whether a row represents a query or a key.


In [10]:
# S10: Deterministic checkpoint record
checkpoint_21 = {
    "one_query_scores": [round(float(value), 4) for value in scores],
    "one_query_weights": [round(float(value), 4) for value in weights],
    "retrieved_value": [round(float(value), 4) for value in retrieved],
    "matrix_score_shape": tuple(all_scores.shape),
    "matrix_output_shape": tuple(self_attention_output.shape),
    "sequential_is_module": isinstance(sequential_mlp, nn.Module),
    "bridge_outputs_match": bool(bridge_outputs_match),
    "custom_parameter_names": custom_parameter_names,
    "custom_parameter_count": custom_parameter_count,
    "residual_output_shape": tuple(residual_output.shape),
    "residual_relation_holds": bool(residual_relation_holds),
    "learned_weight_shape": tuple(learned_weights.shape),
    "learned_output_shape": tuple(learned_output.shape),
}
checkpoint_21


{'one_query_scores': [0.7071, 0.7071, 1.4142],
 'one_query_weights': [0.2483, 0.2483, 0.5035],
 'retrieved_value': [5.5035, 5.5035],
 'matrix_score_shape': (4, 4),
 'matrix_output_shape': (4, 2),
 'sequential_is_module': True,
 'bridge_outputs_match': True,
 'custom_parameter_names': ['first.weight',
  'first.bias',
  'second.weight',
  'second.bias'],
 'custom_parameter_count': 58,
 'residual_output_shape': (1, 4),
 'residual_relation_holds': True,
 'learned_weight_shape': (1, 4, 4),
 'learned_output_shape': (1, 4, 2)}

## Pitfall and extension

**Pitfall.** Softmax must operate across the key positions (`dim=-1`). Applying it down the query dimension answers a different question and does not make each query's weights sum to one.

**Optional extension.** Multiply the query in S2 by 5. The ranking of the keys stays the same, but the softmax becomes sharper; this demonstrates sensitivity to query norm. The separate `sqrt(key_dimension)` factor addresses another effect: when coordinate scales stay fixed, the variance of a dot product grows with the vector dimension.
